# 05 · O sentimento melhora a previsão?

LSTM e Transformer com os hiperparâmetros das etapas 02 e 03, em três versões de entrada:
- **financeiro:** só OHLCV (como nas etapas 02 e 03).
- **sentimento_ffill:** OHLCV + sentimento da etapa 04, dias vazios com o último valor da própria moeda.
- **sentimento_btc:** OHLCV + sentimento da etapa 04, dias vazios com o valor do BTC no mesmo dia.

O sentimento entra com 4 colunas: SentimentoNoticia, SentimentoTweet, TemNoticia e TemTweet.
Mesmo alvo (retorno), mesmo teste, 5 sementes. Diebold-Mariano de cada versão com sentimento contra a financeira.

In [ ]:
import os, sys, glob
achados = glob.glob('../src/tcc/config.py') + glob.glob('/kaggle/input/**/src/tcc/config.py', recursive=True)
RAIZ = achados[0].rsplit('/src/', 1)[0]              # pasta do projeto (no PC ou no dataset do Kaggle)
sys.path.insert(0, os.path.join(RAIZ, 'src'))
import warnings; warnings.filterwarnings('ignore')
import json, shutil
import pandas as pd, matplotlib.pyplot as plt
from tcc import config, experimento, sentimento

In [ ]:
ALVO   = 'retorno'
INICIO = '2022-01-01'
FIM    = '2023-12-31'
MOEDAS = config.MOEDAS
VARIANTES = {'financeiro': None, 'sentimento_ffill': 'ffill', 'sentimento_btc': 'btc'}

ler_hp = lambda etapa: json.load(open(os.path.join(RAIZ, 'results', f'{etapa}_{ALVO}_{INICIO}_{FIM}', 'melhores_hp.json')))
MODELOS = {'LSTM': ler_hp('02_optuna_lstm'), 'Transformer': ler_hp('03_optuna_transformer')}

SAIDA = config.PASTA_RESULTADOS / f'05_sentimento_{ALVO}_{INICIO}_{FIM}'
SAIDA.mkdir(parents=True, exist_ok=True)
MODELOS

## Treino e teste

In [ ]:
partes = []
for m in MOEDAS:
    for variante, metodo in VARIANTES.items():
        extras = None if metodo is None else sentimento.carregar(m, metodo, INICIO, FIM)
        for nome, hp in MODELOS.items():
            partes.append(experimento.rodar(m, ALVO, INICIO, FIM, hp=hp, extras=extras).assign(modelo=nome, variante=variante))
            print(m, variante, nome, 'ok')
prev = pd.concat(partes, ignore_index=True)
prev.to_csv(SAIDA / 'previsoes.csv', index=False)

## MAPE (%)

In [ ]:
resumo = experimento.resumir(prev, chave=['modelo', 'variante'])
resumo.round(4).to_csv(SAIDA / 'resumo.csv', index=False)
tabela = resumo.pivot_table(index='moeda', columns=['modelo', 'variante'], values='MAPE_%').loc[MOEDAS]
tabela = tabela[[(mo, v) for mo in MODELOS for v in VARIANTES]]
tabela.insert(0, ('naive', ''), resumo.groupby('moeda')['MAPE_naive_%'].first())
tabela.loc['média'] = tabela.mean()
tabela.round(3).to_csv(SAIDA / 'tabela_mape.csv')
tabela.round(3)

## Diebold-Mariano: com sentimento x financeiro

DM < 0: a versão com sentimento erra menos. p < 0,05: diferença significativa.

In [ ]:
dm = pd.concat([experimento.comparar(prev[prev.modelo == nome], 'financeiro', 'variante', ['sentimento_ffill', 'sentimento_btc'])
                .assign(modelo=nome) for nome in MODELOS], ignore_index=True)
dm.round(4).to_csv(SAIDA / 'dm_sentimento_vs_financeiro.csv', index=False)
dm.pivot_table(index='moeda', columns=['modelo', 'variante'], values=['DM', 'p']).loc[MOEDAS].round(3)

## Contra o naive

In [ ]:
resumo.pivot_table(index='moeda', columns=['modelo', 'variante'], values=['p_vs_naive', 'acerto_direcao_%']).loc[MOEDAS].round(3)

In [ ]:
fig, eixos = plt.subplots(len(MOEDAS), 1, figsize=(11, 2.6 * len(MOEDAS)))
media = prev.groupby(['moeda', 'modelo', 'variante', 'data'])[['real', 'previsto']].mean().reset_index()
for ax, m in zip(eixos, MOEDAS):
    g = media[(media.moeda == m) & (media.modelo == 'Transformer')]
    ax.plot(g[g.variante == 'financeiro'].data, g[g.variante == 'financeiro'].real, 'k', lw=1.5, label='real')
    for v in VARIANTES:
        h = g[g.variante == v]
        ax.plot(h.data, h.previsto, lw=1, label=v)
    ax.set_ylabel(m); ax.grid(alpha=.3)
eixos[0].legend(ncol=4); eixos[0].set_title('Transformer no teste (média de 5 sementes)')
plt.tight_layout(); plt.savefig(SAIDA / 'previsoes_teste.png', dpi=130); plt.show()

In [ ]:
(SAIDA / 'config.json').write_text(json.dumps({'alvo': ALVO, 'inicio': INICIO, 'fim': FIM, 'moedas': MOEDAS,
    'sementes': config.SEMENTES, 'variantes': VARIANTES, 'colunas_sentimento': sentimento.COLUNAS, 'hp': MODELOS}, indent=2))
print('Pronto:', shutil.make_archive(str(SAIDA), 'zip', SAIDA))